# SDXL Orthographic Hallucination — Baseline & Mechanistic Experiments

This notebook implements:
- **Phase 0**: Controlled prompt suite + OCR evaluation pipeline (baseline CER/WER table)
- **Exp 1**: Text-encoding bottleneck analysis (tokenization, embedding separability, denoiser sensitivity)
- **Exp 2**: Cross-attention misbinding analysis (attention hooks, ROI statistics, scene complexity ablation)

**Runtime requirement**: GPU (T4 minimum, A100 recommended for faster generation). Enable via Runtime → Change runtime type → GPU.

---
## 0. Installation & Imports

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')

In [ ]:
# # Core torch stack — matched versions
# !pip install torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121 --force-reinstall

# # HuggingFace stack - show output for these critical ones
# !pip install huggingface_hub==0.25.2 transformers==4.46.1 accelerate==0.34.2 diffusers==0.31.0 peft==0.14.0 --force-reinstall
# # Other dependencies
# !pip install xformers==0.0.28.post3 --index-url https://download.pytorch.org/whl/cu121 --force-reinstall
# !pip install numpy==1.26.4 surya-ocr==0.4.3 python-Levenshtein pandas matplotlib --force-reinstall



# 1. Uninstall the current conflicting versions to clear the cache
!pip uninstall -y torch torchvision torchaudio xformers

# 2. Install the 'Golden Trio' for Torch 2.5.1
!pip install torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121

# 3. Install xformers built specifically for Torch 2.5.1
!pip install xformers==0.0.28.post3 --index-url https://download.pytorch.org/whl/cu121

# 4. Ensure Diffusers and PEFT match this modern stack
!pip install huggingface_hub==0.25.2 transformers==4.46.1 accelerate==0.34.2 diffusers==0.31.0 peft==0.14.0

In [ ]:
!pip install pandas matplotlib scipy python-Levenshtein

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from pathlib import Path
import huggingface_hub
import diffusers
from itertools import product
from collections import defaultdict
from scipy import stats
import json, os, re
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

from Levenshtein import distance as levenshtein_distance

# Diffusers
from diffusers import StableDiffusionXLPipeline, DDIMScheduler
from diffusers.models.attention_processor import AttnProcessor2_0

# Transformers
from transformers import CLIPTokenizer, CLIPTextModel

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Output directory
# Save to Google Drive for persistence
OUTPUT_DIR = Path('/content/drive/MyDrive/sdxl_orthographic_hallucination_2 (1)')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
(OUTPUT_DIR / 'images').mkdir(exist_ok=True)
(OUTPUT_DIR / 'attention_maps').mkdir(exist_ok=True)
(OUTPUT_DIR / 'results').mkdir(exist_ok=True)
print('Output directories created.')

---
## Phase 0: Prompt Suite & Baseline Pipeline

### 0.1 Define the Factorial Prompt Suite

In [ ]:
# ── Target strings across difficulty axes ──────────────────────────────────────
TARGET_STRINGS = {
    'short_common':    ['CAT', 'DOG', 'SUN'],
    'medium_common':   ['MARKET', 'BRIDGE', 'GARDEN'],
    'long_common':     ['STRAWBERRY', 'THUNDERBOLT', 'WATERMELON'],
    'digits':          ['2847', '90312', '15503'],
    'mixed_alphanum':  ['H2O', 'R2D2', 'B52'],
}

# Flatten to a list of (string, category) tuples
ALL_TARGETS = [
    (s, cat)
    for cat, strings in TARGET_STRINGS.items()
    for s in strings
]

# ── Scene complexity ───────────────────────────────────────────────────────────
SCENE_TEMPLATES = {
    'blank':    'a plain white sign with the word "{target}" printed clearly on it',
    'simple':   'a wooden storefront sign that clearly says "{target}"',
    'cluttered':'a neon sign on a busy city street that reads "{target}"',
}

# ── Prompt format variants ─────────────────────────────────────────────────────
# Each is a function that takes (scene_desc, target) and returns a full prompt
PROMPT_FORMATS = {
    'quoted':    lambda scene, t: scene.replace(f'"{t}"', f'"{t}"'),
    'unquoted':  lambda scene, t: scene.replace(f'"{t}"', t),
    'letterspaced': lambda scene, t: scene.replace(f'"{t}"', ' '.join(list(t))),
    'verbatim':  lambda scene, t: scene.replace(f'"{t}"', t) + f', render the exact characters: {t}',
}


# ── Seeds ──────────────────────────────────────────────────────────────────────
SEEDS = [42, 123]

# ── Build prompt grid ──────────────────────────────────────────────────────────
def build_prompt_grid(targets=ALL_TARGETS,
                      scenes=SCENE_TEMPLATES,
                      formats=PROMPT_FORMATS,
                      seeds=SEEDS):
    """
    Returns a list of dicts, each representing one generation job.
    For the full factorial design this can be large; we provide a
    `subset` flag to run a quick smoke-test first.
    """
    grid = []
    for (target, target_cat), (scene_name, scene_tmpl), (fmt_name, fmt_fn) in product(
        targets, scenes.items(), formats.items()
    ):
        scene_desc = scene_tmpl.format(target=target)
        prompt = fmt_fn(scene_desc, target)
        for seed in seeds:
            grid.append({
                'target':      target,
                'target_cat':  target_cat,
                'scene':       scene_name,
                'format':      fmt_name,
                'seed':        seed,
                'prompt':      prompt,
                'image_path':  None,  # filled in after generation
            })
    return grid


def build_smoke_test_grid():
    """Minimal grid for quick testing: 2 targets × 2 scenes × 1 format × 1 viewpoint × 2 seeds."""
    return build_prompt_grid(
        targets=[('MARKET', 'medium_common'), ('2847', 'digits')],
        scenes={k: SCENE_TEMPLATES[k] for k in ['blank', 'cluttered']},
        formats={k: PROMPT_FORMATS[k] for k in ['quoted']},
        seeds=[42, 123],
    )


# Choose which grid to run:
#   smoke_test_grid  → 8 images, fast, for debugging
#   full_grid        → ~360 images, use for real experiments - can increase if needed
SMOKE_TEST = False   # ← set to False for full run

prompt_grid = build_smoke_test_grid() if SMOKE_TEST else build_prompt_grid()
print(f'Prompt grid size: {len(prompt_grid)} generations')
pd.DataFrame(prompt_grid[:5])

### 0.2 Load SDXL

In [ ]:
# Load SDXL base model
# fp16 + xformers keeps VRAM under 12 GB on T4
MODEL_ID = 'stabilityai/stable-diffusion-xl-base-1.0'

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

# Enable memory-efficient attention if xformers is available
try:
    pipe.enable_xformers_memory_efficient_attention()
    print('xformers enabled')
except Exception:
    print('xformers not available, using default attention')

# Fixed generation settings — never change these across experiments
GEN_KWARGS = dict(
    height=1024,
    width=1024,
    num_inference_steps=50,
    guidance_scale=7.5,
    negative_prompt='blurry, low quality, distorted',
)

print('SDXL loaded.')

### 0.3 Generate Images

In [ ]:
def generate_image(pipe, prompt, seed, save_path=None):
    """Generate a single image with a fixed seed and save it."""
    generator = torch.Generator(device=device).manual_seed(seed)
    with torch.inference_mode():
        result = pipe(prompt=prompt, generator=generator, **GEN_KWARGS)
    image = result.images[0]
    if save_path:
        image.save(save_path)
    return image


# Run generation over the prompt grid
# Skips already-generated images so you can resume interrupted runs
for i, entry in enumerate(prompt_grid):
    fname = f"{entry['target']}_{entry['scene']}_{entry['format']}_seed{entry['seed']}.png"
    save_path = OUTPUT_DIR / 'images' / fname
    entry['image_path'] = str(save_path)

    if save_path.exists():
        print(f'[{i+1}/{len(prompt_grid)}] Skipping (exists): {fname}')
        continue

    print(f'[{i+1}/{len(prompt_grid)}] Generating: {fname}')
    generate_image(pipe, entry['prompt'], entry['seed'], save_path=save_path)

print('\nAll generations complete.')

### 0.4 OCR Evaluation Pipeline

In [ ]:
# trocr_processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
# trocr_model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed').to(device)
# trocr_model.eval()
# print('TrOCR ready.')

# def run_ocr(image_path):
#     """
#     Returns list of (bbox, text, confidence) tuples
#     """
#     image = Image.open(image_path).convert('RGB')
#     pixel_values = trocr_processor(image, return_tensors='pt').pixel_values.to(device)
#     with torch.no_grad():
#         generated_ids = trocr_model.generate(pixel_values)
#     text = trocr_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
#     if not text:
#         return []
#     # Return in same (bbox, text, confidence) format score_image expects
#     return [([[0,0],[0,0],[0,0],[0,0]], text, 1.0)]


In [ ]:
!pip install easyocr

In [ ]:
import easyocr

ocr_reader = easyocr.Reader(['en'], gpu=True)
print('EasyOCR ready.')

def run_ocr(image_path):
    # detail=1 returns (bbox, text, confidence) — already the format score_image expects
    results = ocr_reader.readtext(str(image_path), detail=1)
    return results if results else []

In [ ]:
# ── Metric functions ───────────────────────────────────────────────────────────

def ned(detected, target):
    """Normalized Edit Distance ∈ [0, 1]. 0 = perfect match."""
    if len(detected) == 0 and len(target) == 0:
        return 0.0
    return levenshtein_distance(detected.upper(), target.upper()) / max(len(detected), len(target))


def cer(detected, target):
    """Character Error Rate = edit_distance / len(target). Can exceed 1."""
    if len(target) == 0:
        return 0.0 if len(detected) == 0 else 1.0
    return levenshtein_distance(detected.upper(), target.upper()) / len(target)


def wer(detected, target):
    """Word Error Rate at word level."""
    d_words = detected.upper().split()
    t_words = target.upper().split()
    if len(t_words) == 0:
        return 0.0 if len(d_words) == 0 else 1.0
    return levenshtein_distance(d_words, t_words) / len(t_words)


def classify_failure(detected, target, ned_score):
    """Classify the type of orthographic hallucination."""
    if detected == '':
        return 'no_text'
    if ned_score == 0.0:
        return 'success'
    if ned_score <= 0.1:
        return 'near_miss'
    # Check for visually similar glyph substitutions
    glyph_subs = {'O': '0', '0': 'O', 'I': '1', '1': 'I', 'E': '3', 'S': '5', 'B': '8'}
    cleaned = detected.upper()
    for orig, sub in glyph_subs.items():
        cleaned = cleaned.replace(sub, orig)
    if ned(cleaned, target) <= 0.1:
        return 'glyph_substitution'
    if ned_score > 0.8:
        return 'garbled'
    return 'partial'



# ── Per-image OCR scoring ──────────────────────────────────────────────────────

def score_image(image_path, target, confidence_threshold=0.3):
    """
    Run OCR on an image and return a dict of metrics.
    Handles no-detection and multi-detection cases.
    """
    results = run_ocr(image_path)

    if not results:
        return {
            'detected_string': '',
            'ocr_confidence':  0.0,
            'ned':             1.0,
            'cer':             1.0,
            'wer':             1.0,
            'success_strict':  False,
            'success_lenient': False,
            'failure_type':    'no_text',
            'n_detections':    0,
            'spurious_text':   False,
        }

    # Sort detections top-to-bottom, left-to-right
    results_sorted = sorted(results, key=lambda r: (r[0][0][1], r[0][0][0]))

    # Among all detections, find the one with best NED against target
    best_ned = 1.0
    best_str = ''
    best_conf = 0.0

    # Also try concatenating all detections
    concat_str = ''.join(r[1] for r in results_sorted if r[2] >= confidence_threshold)

    candidates = [(r[1], r[2]) for r in results_sorted] + [(concat_str, np.mean([r[2] for r in results_sorted]))]

    for cand_str, cand_conf in candidates:
        n = ned(cand_str, target)
        if n < best_ned:
            best_ned = n
            best_str = cand_str
            best_conf = cand_conf

    # Spurious text: any high-confidence detection that doesn't match target
    spurious = any(
        r[2] >= confidence_threshold and ned(r[1], target) > 0.5
        for r in results_sorted
    )

    ned_score = best_ned
    return {
        'detected_string': best_str,
        'ocr_confidence':  float(best_conf),
        'ned':             float(ned_score),
        'cer':             float(cer(best_str, target)),
        'wer':             float(wer(best_str, target)),
        'success_strict':  best_str.upper().strip() == target.upper().strip(),
        'success_lenient': ned_score <= 0.1,
        'failure_type':    classify_failure(best_str, target, ned_score),
        'n_detections':    len(results),
        'spurious_text':   spurious,
    }


print('Metric functions defined.')

In [ ]:
!pip install gdown -q
!gdown --folder https://drive.google.com/drive/folders/1gJetCGMfFuzJMIBsfW2PdNTWHQmezumL \
    -O /content/sdxl_data/ --remaining-ok

In [ ]:
OUTPUT_DIR = Path("/content/drive/MyDrive/sdxl_orthographic_hallucination_2 (1)")

images = list(OUTPUT_DIR.rglob('*.png'))
print(f"Found {len(images)} images")
print(images[:3])

(OUTPUT_DIR / 'results').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'images').mkdir(parents=True, exist_ok=True)

In [ ]:
# to avoid regeneration
prompt_grid = build_prompt_grid()  # swap for build_prompt_grid() for full grid

for entry in prompt_grid:
    fname = f"{entry['target']}_{entry['scene']}_{entry['format']}_seed{entry['seed']}.png"
    matches = list(OUTPUT_DIR.rglob(fname))
    entry['image_path'] = str(matches[0]) if matches else None

found = sum(1 for e in prompt_grid if e['image_path'] is not None)
print(f"{found}/{len(prompt_grid)} images matched")


In [ ]:
# Select the first 5 entries from prompt_grid for a quick test
test_grid_subset = prompt_grid[:5]
test_results_records = []

print(f"Running OCR evaluation on {len(test_grid_subset)} images:")

for i, entry in enumerate(test_grid_subset):
    # image_path should already be populated from the previous generation step (cell JFg1GA2GVico)
    if entry.get('image_path') is None or not Path(entry['image_path']).exists():
        print(f"  Warning: Image not found for {entry['target']} (seed {entry['seed']}). It might not have been generated yet or path is incorrect. Skipping evaluation for this entry.")
        continue

    # Extract filename for printing
    fname = Path(entry['image_path']).name
    print(f"[{i+1}/{len(test_grid_subset)}] Evaluating: {fname}")

    scores = score_image(entry['image_path'], entry['target'])
    record = {**entry, **scores}
    test_results_records.append(record)

    print(f"  Target: {entry['target']:<10} Detected: '{scores['detected_string']:<15}' CER: {scores['cer']:.2f} Type: {scores['failure_type']}")

print("\n--- Test Results ---")
if test_results_records:
    test_df = pd.DataFrame(test_results_records)
    # Display relevant columns, dropping 'prompt' as it can be very long
    display_cols = ['target', 'scene', 'format', 'detected_string', 'cer', 'failure_type', 'success_lenient']
    print(test_df[display_cols].to_string())

    # Optionally save this test subset results
    test_df.to_csv(OUTPUT_DIR / 'results' / 'test_subset_scores_quick.csv', index=False)
    print(f"\nTest subset results saved to: {OUTPUT_DIR / 'results' / 'test_subset_scores_quick.csv'}")
else:
    print("No images were evaluated. Please ensure images have been generated by running the previous cell.")

In [ ]:
# Run OCR evaluation over all generated images
results_records = []

for i, entry in enumerate(prompt_grid):
    if entry['image_path'] is None or not Path(entry['image_path']).exists():
        print(f'[{i+1}] Missing image, skipping: {entry}')
        continue

    scores = score_image(entry['image_path'], entry['target'])
    record = {**entry, **scores}
    results_records.append(record)

    print(f"[{i+1}/{len(prompt_grid)}] {entry['target']:15s} "
          f"scene={entry['scene']:10s} fmt={entry['format']:12s} "
          f"detected='{scores['detected_string']:15s}' "
          f"CER={scores['cer']:.2f} type={scores['failure_type']}")

# Save results
results_df = pd.DataFrame(results_records)
results_df.to_csv(OUTPUT_DIR / 'results' / 'baseline_scores.csv', index=False)
print(f'\nResults saved. Shape: {results_df.shape}')

### 0.5 Baseline Visualization

In [ ]:
def bootstrap_ci(values, n_boot=1000, ci=95):
    """Return (mean, lower_ci, upper_ci) via bootstrapping."""
    values = np.array(values)
    boot_means = [np.mean(np.random.choice(values, size=len(values), replace=True)) for _ in range(n_boot)]
    lo = np.percentile(boot_means, (100 - ci) / 2)
    hi = np.percentile(boot_means, 100 - (100 - ci) / 2)
    return np.mean(values), lo, hi


fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Baseline CER by Experimental Axis', fontsize=14, fontweight='bold')

# ── Plot 1: CER by target category ────────────────────────────────────────────
ax = axes[0]
cat_order = ['short_common', 'medium_common', 'long_common', 'digits', 'mixed_alphanum']
cat_stats = []
for cat in cat_order:
    subset = results_df[results_df['target_cat'] == cat]['cer'].values
    if len(subset) > 0:
        mean, lo, hi = bootstrap_ci(subset)
        cat_stats.append({'cat': cat.replace('_', '\n'), 'mean': mean, 'lo': lo, 'hi': hi})

cat_df = pd.DataFrame(cat_stats)
bars = ax.bar(cat_df['cat'], cat_df['mean'], color='steelblue', alpha=0.8)
ax.errorbar(range(len(cat_df)), cat_df['mean'],
            yerr=[cat_df['mean'] - cat_df['lo'], cat_df['hi'] - cat_df['mean']],
            fmt='none', color='black', capsize=4)
ax.set_ylabel('Mean CER')
ax.set_title('By String Category')
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.4, label='CER=0.5')
ax.legend(fontsize=8)

# ── Plot 2: CER by scene complexity ───────────────────────────────────────────
ax = axes[1]
scene_stats = []
for scene in ['blank', 'simple', 'cluttered']:
    subset = results_df[results_df['scene'] == scene]['cer'].values
    if len(subset) > 0:
        mean, lo, hi = bootstrap_ci(subset)
        scene_stats.append({'scene': scene, 'mean': mean, 'lo': lo, 'hi': hi})

scene_df = pd.DataFrame(scene_stats)
colors = ['#2ecc71', '#f39c12', '#e74c3c']
ax.bar(scene_df['scene'], scene_df['mean'], color=colors, alpha=0.8)
ax.errorbar(range(len(scene_df)), scene_df['mean'],
            yerr=[scene_df['mean'] - scene_df['lo'], scene_df['hi'] - scene_df['mean']],
            fmt='none', color='black', capsize=4)
ax.set_ylabel('Mean CER')
ax.set_title('By Scene Complexity')
ax.set_ylim(0, 1.05)

# ── Plot 3: CER by prompt format ──────────────────────────────────────────────
ax = axes[2]
fmt_stats = []
for fmt in ['quoted', 'unquoted', 'letterspaced', 'verbatim']:
    subset = results_df[results_df['format'] == fmt]['cer'].values
    if len(subset) > 0:
        mean, lo, hi = bootstrap_ci(subset)
        fmt_stats.append({'fmt': fmt, 'mean': mean, 'lo': lo, 'hi': hi})

fmt_df = pd.DataFrame(fmt_stats)
ax.bar(fmt_df['fmt'], fmt_df['mean'], color='mediumpurple', alpha=0.8)
ax.errorbar(range(len(fmt_df)), fmt_df['mean'],
            yerr=[fmt_df['mean'] - fmt_df['lo'], fmt_df['hi'] - fmt_df['mean']],
            fmt='none', color='black', capsize=4)
ax.set_ylabel('Mean CER')
ax.set_title('By Prompt Format')
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'results' / 'baseline_cer_by_axis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Baseline plots saved.')

In [ ]:
# Failure type distribution heatmap: rows=target_cat, cols=failure_type
failure_pivot = (
    results_df.groupby(['target_cat', 'failure_type'])
    .size()
    .unstack(fill_value=0)
)
# Normalize to proportions per row
failure_pivot_pct = failure_pivot.div(failure_pivot.sum(axis=1), axis=0)

plt.figure(figsize=(10, 5))
sns.heatmap(failure_pivot_pct, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Proportion'})
plt.title('Failure Type Distribution by Target Category')
plt.ylabel('Target Category')
plt.xlabel('Failure Type')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'results' / 'failure_type_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# 1. Text Encoding Bottleneck

### 1.1 Tokenization analysis

In [ ]:
tokenizer_1 = pipe.tokenizer      # CLIP ViT-L
tokenizer_2 = pipe.tokenizer_2    # OpenCLIP ViT-bigG

target_cer = results_df.groupby('target').agg(
    mean_cer=('cer', 'mean'),
    n_samples=('cer', 'count'),
    target_cat=('target_cat', 'first')
).reset_index()

print(f"Unique targets: {len(target_cer)}")
print(target_cer[['target', 'target_cat', 'mean_cer', 'n_samples']].to_string())

In [ ]:
frag_records = []

for _, row in target_cer.iterrows():
    target = row['target']
    n_chars = len(target)

    tok1 = tokenizer_1.encode(target)
    tok2 = tokenizer_2.encode(target)

    content_tok1 = [t for t in tok1 if t not in tokenizer_1.all_special_ids]
    content_tok2 = [t for t in tok2 if t not in tokenizer_2.all_special_ids]

    n_tok1 = len(content_tok1)
    n_tok2 = len(content_tok2)

    frag_records.append({
        'target': target,
        'target_cat': row['target_cat'],
        'n_chars': n_chars,
        'n_tokens_clip': n_tok1,
        'n_tokens_openclip': n_tok2,
        'frag_clip': n_tok1 / n_chars,
        'frag_openclip': n_tok2 / n_chars,
        'frag_mean': (n_tok1 + n_tok2) / (2 * n_chars),
        'mean_cer': row['mean_cer'],
        'subtokens_clip': tokenizer_1.convert_ids_to_tokens(content_tok1),
        'subtokens_openclip': tokenizer_2.convert_ids_to_tokens(content_tok2),
    })

frag_df = pd.DataFrame(frag_records)

for _, row in frag_df.iterrows():
    print(f"\n{row['target']!r} ({row['n_chars']} chars, CER={row['mean_cer']:.3f})")
    print(f"  CLIP:     {row['subtokens_clip']}")
    print(f"  OpenCLIP: {row['subtokens_openclip']}")
    print(f"  Frag: {row['frag_clip']:.3f} / {row['frag_openclip']:.3f}")

In [ ]:
r, p = stats.pearsonr(frag_df['frag_mean'], frag_df['mean_cer'])
rho, p_s = stats.spearmanr(frag_df['frag_mean'], frag_df['mean_cer'])
print(f"Pearson  r={r:.4f}  p={p:.4f}")
print(f"Spearman rho={rho:.4f}  p={p_s:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))

cats = frag_df['target_cat'].unique()
colors = {c: plt.cm.Set2(i) for i, c in enumerate(cats)}

for cat in cats:
    mask = frag_df['target_cat'] == cat
    ax.scatter(frag_df.loc[mask, 'frag_mean'], frag_df.loc[mask, 'mean_cer'],
               c=[colors[cat]], label=cat.replace('_', ' '), s=80,
               edgecolors='black', linewidth=0.5)

slope, intercept = np.polyfit(frag_df['frag_mean'], frag_df['mean_cer'], 1)
x_line = np.linspace(0, 1.1, 100)
ax.plot(x_line, slope * x_line + intercept, 'r--', alpha=0.6)

ax.set_xlabel('Mean Fragmentation Score (tokens / chars)')
ax.set_ylabel('Mean CER')
ax.set_title(f'Fragmentation vs CER (r={r:.3f}, p={p:.3f})')
ax.legend(fontsize=8)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'results' / 'exp1_1_fragmentation_vs_cer.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.2 Embedding separability

In [ ]:
def get_embedding(text, tokenizer, encoder):
    """Get pooled text embedding for a string."""
    tokens = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    tokens = {k: v.to(encoder.device) for k, v in tokens.items()}
    with torch.no_grad():
        out = encoder(**tokens)
    # Use last hidden state, mean-pooled
    return out.last_hidden_state.mean(dim=1).squeeze().cpu().float()

def make_neighbors(target):
    """Generate minimal-edit neighbors: substitutions, insertions, deletions."""
    neighbors = []
    chars = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'

    # Substitutions: swap each character for something else
    for i in range(len(target)):
        for c in chars:
            if c != target[i]:
                new = target[:i] + c + target[i+1:]
                neighbors.append(('sub', i, new))
                break  # just one substitution per position

    # Deletions: remove each character
    for i in range(len(target)):
        new = target[:i] + target[i+1:]
        neighbors.append(('del', i, new))

    # Insertions: add a character at each position
    for i in range(len(target) + 1):
        new = target[:i] + 'X' + target[i:]
        neighbors.append(('ins', i, new))

    return neighbors

In [ ]:
import torch.nn.functional as F

# Get text encoders
text_encoder_1 = pipe.text_encoder    # CLIP ViT-L
text_encoder_2 = pipe.text_encoder_2  # OpenCLIP ViT-bigG

# Compute embeddings and distances
sep_records = []

for target in target_cer['target'].values:
    neighbors = make_neighbors(target)

    emb1_target = get_embedding(target, tokenizer_1, text_encoder_1)
    emb2_target = get_embedding(target, tokenizer_2, text_encoder_2)

    for edit_type, pos, neighbor in neighbors:
        emb1_neighbor = get_embedding(neighbor, tokenizer_1, text_encoder_1)
        emb2_neighbor = get_embedding(neighbor, tokenizer_2, text_encoder_2)

        cos1 = F.cosine_similarity(emb1_target.unsqueeze(0), emb1_neighbor.unsqueeze(0)).item()
        cos2 = F.cosine_similarity(emb2_target.unsqueeze(0), emb2_neighbor.unsqueeze(0)).item()

        sep_records.append({
            'target': target,
            'neighbor': neighbor,
            'edit_type': edit_type,
            'edit_pos': pos,
            'cos_clip': cos1,
            'cos_openclip': cos2,
            'cos_mean': (cos1 + cos2) / 2,
            'dist_clip': 1 - cos1,
            'dist_openclip': 1 - cos2,
        })

sep_df = pd.DataFrame(sep_records)
print(f"Computed {len(sep_df)} pairwise distances across {len(target_cer)} targets")

In [ ]:
# Summary stats per target
sep_summary = sep_df.groupby('target').agg(
    mean_cos_clip=('cos_clip', 'mean'),
    mean_cos_openclip=('cos_openclip', 'mean'),
    mean_cos=('cos_mean', 'mean'),
).reset_index()

sep_summary = sep_summary.merge(target_cer[['target', 'mean_cer', 'target_cat']], on='target')

print("Mean cosine similarity between target and its 1-edit neighbors:")
print(sep_summary[['target', 'target_cat', 'mean_cos_clip', 'mean_cos_openclip', 'mean_cer']].to_string())

# Correlation: high similarity (hard to distinguish) -> high CER?
r, p = stats.pearsonr(sep_summary['mean_cos'], sep_summary['mean_cer'])
rho, p_s = stats.spearmanr(sep_summary['mean_cos'], sep_summary['mean_cer'])
print(f"\nPearson  r={r:.4f}  p={p:.4f}")
print(f"Spearman rho={rho:.4f}  p={p_s:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

cats = sep_summary['target_cat'].unique()
colors = {c: plt.cm.Set2(i) for i, c in enumerate(cats)}

for cat in cats:
    mask = sep_summary['target_cat'] == cat
    ax.scatter(sep_summary.loc[mask, 'mean_cos'], sep_summary.loc[mask, 'mean_cer'],
               c=[colors[cat]], label=cat.replace('_', ' '), s=80,
               edgecolors='black', linewidth=0.5)

slope, intercept = np.polyfit(sep_summary['mean_cos'], sep_summary['mean_cer'], 1)
x_line = np.linspace(sep_summary['mean_cos'].min() - 0.01, sep_summary['mean_cos'].max() + 0.01, 100)
ax.plot(x_line, slope * x_line + intercept, 'r--', alpha=0.6)

ax.set_xlabel('Mean Cosine Similarity (target vs 1-edit neighbors)')
ax.set_ylabel('Mean CER')
ax.set_title(f'Embedding Separability vs CER (r={r:.3f}, p={p:.3f})')
ax.legend(fontsize=8)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'results' / 'exp1_2_separability_vs_cer.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.3 Denoiser sensitivity

In [ ]:
def get_noise_pred(pipe, prompt, seed, timestep_idx=25):
    """
    Run SDXL up to a specific timestep and return the predicted noise.
    timestep_idx: which step to capture (0=first, 49=last for 50 steps)
    """
    g = torch.Generator(device='cpu').manual_seed(seed)
    latents = pipe.prepare_latents(
        1, pipe.unet.config.in_channels, 1024, 1024,
        torch.float16, pipe.device, generator=g
    )

    pipe.scheduler.set_timesteps(50, device=pipe.device)
    timesteps = pipe.scheduler.timesteps

    # Encode prompt
    prompt_embeds, negative_embeds, pooled_prompt_embeds, negative_pooled = pipe.encode_prompt(
        prompt=prompt,
        device=pipe.device,
        num_images_per_prompt=1,
        do_classifier_free_guidance=True,
        negative_prompt='blurry, low quality, distorted',
    )

    add_text_embeds = pooled_prompt_embeds
    add_time_ids = pipe._get_add_time_ids(
        (1024, 1024), (0, 0), (1024, 1024),
        dtype=prompt_embeds.dtype,
        text_encoder_projection_dim=pipe.text_encoder_2.config.projection_dim,
    )
    add_time_ids = add_time_ids.to(pipe.device)

    # Denoise up to target timestep
    for i, t in enumerate(timesteps):
        if i > timestep_idx:
            break

        latent_input = torch.cat([latents] * 2)
        latent_input = pipe.scheduler.scale_model_input(latent_input, t)

        added_cond = {
            "text_embeds": torch.cat([negative_pooled, add_text_embeds]),
            "time_ids": torch.cat([add_time_ids] * 2),
        }

        with torch.no_grad():
            noise_pred = pipe.unet(
                latent_input, t,
                encoder_hidden_states=torch.cat([negative_embeds, prompt_embeds]),
                added_cond_kwargs=added_cond,
            ).sample

        # CFG
        noise_uncond, noise_text = noise_pred.chunk(2)
        noise_pred = noise_uncond + 7.5 * (noise_text - noise_uncond)

        latents = pipe.scheduler.step(noise_pred, t, latents).prev_sample

    return noise_pred.detach().cpu().float()

In [ ]:
pairs = [
    ('WATER', 'WAFER'),     # medium common
    ('DOG', 'DIG'),         # short common
    ('CAT', 'CAR'),         # short common
    ('2847', '2147'),       # digits
    ('R2D2', 'R2D3'),       # mixed
    ('GARDEN', 'WARDEN'),   # medium common
]

template = 'a plain white sign with the word "{}" printed clearly on it'
seed = 42

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, (word_a, word_b) in zip(axes.flat, pairs):
    prompt_a = template.format(word_a)
    prompt_b = template.format(word_b)

    noise_a = get_noise_pred(pipe, prompt_a, seed)
    noise_b = get_noise_pred(pipe, prompt_b, seed)

    diff = (noise_a - noise_b).squeeze(0)
    l2_map = diff.pow(2).sum(dim=0).sqrt()

    ax.imshow(l2_map.numpy(), cmap='hot')
    ax.set_title(f'{word_a} vs {word_b}')
    ax.axis('off')

plt.suptitle('Denoiser Sensitivity Heatmaps', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'results' / 'exp1_3_heatmap_grid.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.imshow(l2_map.numpy(), cmap='hot')
plt.colorbar(label='L2 difference')
plt.title('WATER vs WAFER: where does the denoiser respond?')
plt.savefig(OUTPUT_DIR / 'results' / 'exp1_3_l2_heatmap.png', dpi=150)
plt.show()

In [ ]:
g = torch.Generator(device='cpu').manual_seed(42)
img = pipe(prompt_a, generator=g, **GEN_KWARGS).images[0]
display(img.resize((400, 400)))